# LAB-HW-05 — 第一次启动 PS/Linux + UART Console

**今天只新增一件事：** 让 KV260 的 processor system 真正启动 Linux，并证明 runtime host 与 PL programming 是两件不同的事。

前置：LAB-HW-00~04。

**Project Trace:** RMD-012B · T-HW-005/T-HW-011

## 1. development host 不是 runtime host

<svg xmlns="http://www.w3.org/2000/svg" width="900" height="270" viewBox="0 0 900 270" role="img" aria-label="LAB-HW-05 development host UART and KV260 Linux boot path">
  <rect x="25" y="80" width="170" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="110" y="112" text-anchor="middle" font-size="15">development host</text>
  <text x="110" y="136" text-anchor="middle" font-size="12">download / flash / UART</text>
  <rect x="255" y="80" width="125" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="318" y="112" text-anchor="middle" font-size="15">J4 FTDI</text>
  <text x="318" y="136" text-anchor="middle" font-size="12">USB UART</text>
  <rect x="450" y="35" width="180" height="80" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="540" y="68" text-anchor="middle" font-size="15">K26 PS / Linux</text>
  <text x="540" y="92" text-anchor="middle" font-size="12">runtime host</text>
  <rect x="450" y="160" width="180" height="80" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="540" y="193" text-anchor="middle" font-size="15">J11 microSD</text>
  <text x="540" y="217" text-anchor="middle" font-size="12">Ubuntu Server image</text>
  <rect x="700" y="80" width="165" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="782" y="112" text-anchor="middle" font-size="15">shell evidence</text>
  <text x="782" y="136" text-anchor="middle" font-size="12">kernel / OS / board</text>
  <path d="M195 122 L255 122" stroke="#333" stroke-width="2"/><polygon points="255,122 245,117 245,127" fill="#333"/>
  <path d="M380 122 L450 82" stroke="#333" stroke-width="2"/><polygon points="450,82 438,83 443,92" fill="#333"/>
  <path d="M540 160 L540 115" stroke="#333" stroke-width="2"/><polygon points="540,115 535,125 545,125" fill="#333"/>
  <path d="M630 82 L700 122" stroke="#333" stroke-width="2"/><polygon points="700,122 688,112 686,123" fill="#333"/>
</svg>

development host 负责下载/写 image、打开 UART；**runtime host** 是运行在 K26 processor system（PS）上的 Linux。

JTAG program PL 与 Linux boot 是两条不同路径，不要把它们混成“FPGA 启动”。

## 2. 第二阶段 Ubuntu image

课程 authoring candidate：

`iot-limerick-kria-classic-server-2404-classic-24.04-x07-20250423.img.xz`

distribution：AMD Kria K26 starter kit 的 **Ubuntu Server 24.04 LTS**。

来源身份记录在：

`boards/kv260/runtime/ubuntu24_image.json`

重要 evidence boundary：撰写时可见的 Canonical 下载目录没有发布 upstream SHA-256，因此仓库**不会编一个 hash**。你需要计算实际下载文件的 SHA-256 并保存；在课程把一次受控下载的 expected hash 冻结进 manifest 前，不能宣称 formal image-hash PASS。

## 3. 记录实际下载 image 身份

development host 在仓库根目录运行：

```bash
python boards/kv260/runtime/hash_image.py \
  /path/to/iot-limerick-kria-classic-server-2404-classic-24.04-x07-20250423.img.xz
```

当前 authoring stage 预期：

`STATUS=RECORDED_UNVERIFIED`

这是刻意设计。把 filename、byte size、SHA-256 写入 T-HW-011 evidence。

## 4. 写 microSD

使用 **16 GB UHS-1 或更大** microSD。

初学者路径用 AMD 当前 Kria Ubuntu guide 推荐的 **Raspberry Pi Imager**：

1. 选择下载的 image；
2. 选择正确 microSD；
3. Write；
4. 等待完成并安全弹出；
5. 插入 KV260 **J11**。

注意目标磁盘：写 image 会覆盖你选择的设备。

## 5. 上电前先打开 UART console

用支持 data 的 USB cable 把 J4 接到 development host。

UART settings：

- 115200 baud
- 8 data bits
- no parity
- 1 stop bit
- no hardware/software flow control

Windows/macOS 的 AMD guide 把枚举出来的第二个 FTDI serial port 作为 UART。Linux 的设备号可能随机器变化，所以观察 J4 插入前后新增的 FTDI serial device，不要死记某个 `ttyUSB` 数字。

**先开始 terminal logging，再给板上电**，这样 boot transcript 从第一行开始。

### 零基础 terminal 选择

terminal 软件本身不计分，但必须能显式设置上述 UART 参数并保存 log。

- **Windows**：可以用 PuTTY/Tera Term。选择 J4 枚举出的 UART COM port，Connection type 选 Serial，Speed 115200，Flow control 设为 None，并在上电前开启 session logging。
- **Linux**：可以安装 `picocom`（例如 Ubuntu development host：`sudo apt install picocom`），然后使用：

```bash
picocom -b 115200 --flow n --parity n --databits 8 /dev/ttyUSBX
```

其中 `/dev/ttyUSBX` 必须通过插拔 J4 前后设备变化确认，不要照抄示例编号。无论使用哪种 terminal，都先开启日志再上电。


## 6. 上电并进入 shell

J11 microSD、J4 UART 都接好后，再给 J12 接 12 V / 3 A power。

AMD 当前 Ubuntu first-login：

- username：`ubuntu`
- password：`ubuntu`

首次登录按系统要求修改密码。

今天不要装应用、不要 load PL firmware、不要调 AXI。今天的 success condition 只有一个：**PS/runtime host 启动，并进入 shell。**

## 7. 收集 machine-readable boot evidence

运行：

```bash
uname -a
cat /etc/os-release
cat /proc/device-tree/model; echo
sudo xmutil boardid
sudo xmutil bootfw_status
```

或者把仓库 helper 拷到 runtime host 后运行：

```bash
bash boards/kv260/runtime/collect_boot_info.sh
```

输出与 UART boot transcript 一起保留。

## 8. Boot firmware 边界

AMD 明确说明当前 boot firmware 对新 OS compatibility 很重要。本 Lab 主流程先用 `xmutil bootfw_status` **观察**。

如果 Linux 因 firmware 太旧无法启动，再走 AMD 官方 boot-firmware update/recovery troubleshooting branch，并记录修改。

不要把“刷固件试试”变成无证据的常规步骤。

## 9. Clean shutdown

断电前执行：

```bash
sudo shutdown -h now
```

等待 shutdown 完成，再移除 power。

这是保护 microSD filesystem 的正式实验步骤，不是可省略的 housekeeping。

## 10. Expected Evidence / Save Evidence

至少记录：

- image filename、size、本地计算 SHA-256；
- 写卡工具；
- UART device/COM port 与 115200 8N1/no-flow settings；
- 完整 UART boot transcript；
- 首次进入 shell；
- `uname -a`、`/etc/os-release`、device-tree model；
- `xmutil boardid`、`xmutil bootfw_status`；
- board/carrier revision、Git commit、date；
- clean shutdown observation。

复制并填写 `boards/kv260/evidence/manifest.example.json`。

expected image SHA-256 尚未冻结时，不要把 T-HW-005 标成最终 image-hash PASS。

### 学习进度 gate 与正式 acceptance gate

当前 `ubuntu24_image.json` 的 expected SHA-256 仍未冻结，所以 `hash_image.py` 可能正确返回 `STATUS=RECORDED_UNVERIFIED`。这时只要真实板已经满足：UART boot transcript 完整、成功进入 shell、board/OS/kernel identity 已记录、clean shutdown 已完成，就可以满足**学习进度 gate**进入 LAB-HW-06。

这不允许把 T-HW-005 写成正式 PASS。最终 acceptance 仍要等待课程冻结 expected image hash 并完成对应实体 evidence。


## 11. If it does not work

按层级排查：

1. power LEDs 都没有 → 回到 J12/power；
2. 有 power/heartbeat、无 UART → J4 cable、FTDI driver、serial-port selection、115200 8N1/no flow control；
3. UART 有 boot firmware 输出、SD Linux 不启动 → image write、J11 card 或 boot-firmware compatibility；
4. 能 login 但没有 `xmutil` → OS/platform image mismatch；保存 evidence，不要临时拼凑；
5. 强制断电后出现 filesystem warning → 停止硬断电，始终 clean shutdown。

## 12. Human Check

解释：

1. 哪台机器是 development host？
2. runtime host 运行在哪个 processor 上？
3. 为什么 JTAG program PL 成功不能证明 Linux boot？
4. 为什么 image filename 仍不足以唯一标识 artifact？
5. 为什么主流程记录 boot firmware，但不随意更新？
6. 为什么断电前必须 shutdown？

## 13. 官方依据

- AMD UG1089 — Software Getting Started / boot devices / interfaces
- AMD Kria KV260 Ubuntu 24.04 boot guide — SD card、first boot、firmware update
- Canonical — Install Ubuntu on AMD，Kria K26 Ubuntu Server 24.04 LTS
- AMD/Xilinx xmutil — `boardid`、`bootfw_status`

具体 URL 已写入项目文档与 `ubuntu24_image.json`。